# Revenue & Product Analysis

Revenue analysis is where a lot of the business questions live — are we growing, where are the peaks and troughs, what's the actual impact of discounting, and are refunds a problem worth worrying about?

Product analysis layers on top of that: which products are carrying the most revenue, and is the premium tier actually outperforming?


In [ ]:

import sys; sys.path.insert(0, '..')
from pipeline.load import get_connection
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

con = get_connection()
tx = con.execute("SELECT * FROM transactions").df()
pp = con.execute("SELECT * FROM mart_product_performance").df()
tx['timestamp'] = pd.to_datetime(tx['timestamp'])
tx['month'] = tx['timestamp'].dt.to_period('M').astype(str)


## Monthly revenue trend

The dual-axis chart shows orders and revenue together. These should broadly track each other — if they diverge (revenue growing faster than orders), average order value is going up, which is a good sign. If orders are growing but revenue is flat, there might be a discounting problem or a product mix shift towards cheaper items.


In [ ]:

monthly = tx.groupby('month').agg(
    orders=('transaction_id','count'),
    net_revenue=('net_revenue','sum'),
    gross_revenue=('gross_revenue','sum'),
    refunds=('refund_flag','sum'),
    aov=('net_revenue','mean')
).reset_index()
fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()
ax1.bar(monthly['month'], monthly['orders'], color='steelblue', alpha=0.5, label='Orders')
ax2.plot(monthly['month'], monthly['net_revenue'], color='crimson', marker='o', lw=2, label='Net Revenue')
ax1.set_ylabel('Orders'); ax2.set_ylabel('Net Revenue ($)')
ax1.set_title('Monthly Orders & Revenue')
lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc='upper left')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


## Discount impact — are we giving away margin unnecessarily?

Discounts can drive volume but eat into margin. The key question is whether discounted orders have meaningfully higher revenue (suggesting bundles or upsell) or lower revenue (suggesting the discount is just reducing price without changing basket size). If discounted AOV is lower and volume isn't dramatically higher, the discounting strategy needs a rethink.


In [ ]:

tx['discounted'] = tx['discount_applied'] > 0
disc = tx.groupby('discounted').agg(
    orders=('transaction_id','count'),
    avg_revenue=('net_revenue','mean'),
    avg_qty=('quantity','mean')
).round(2)
print(disc)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
disc['avg_revenue'].plot(kind='bar', ax=axes[0], color=['steelblue','coral'], edgecolor='white')
axes[0].set_title('Avg Revenue: Discounted vs Not'); axes[0].set_xticklabels(['No Discount','Discounted'], rotation=0)
disc['orders'].plot(kind='bar', ax=axes[1], color=['steelblue','coral'], edgecolor='white')
axes[1].set_title('Order Count: Discounted vs Not'); axes[1].set_xticklabels(['No Discount','Discounted'], rotation=0)
plt.tight_layout(); plt.show()


## Refund rate over time

A refund rate around 2-3% is fairly normal in e-commerce. Spikes are worth investigating — they often indicate a product quality issue, a fulfilment problem, or a campaign that over-promised. A sustained upward trend is a red flag that something structural has changed.


In [ ]:

refund_monthly = tx.groupby('month')['refund_flag'].agg(['sum','count'])
refund_monthly['refund_rate'] = (refund_monthly['sum'] / refund_monthly['count']).round(4)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(refund_monthly.index, refund_monthly['refund_rate'], color='crimson', marker='o', lw=2)
ax.fill_between(refund_monthly.index, refund_monthly['refund_rate'], alpha=0.2, color='crimson')
ax.set_title('Monthly Refund Rate'); ax.set_ylabel('Refund Rate')
ax.set_xlabel('Month'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
print(f"Overall refund rate: {refund_monthly['sum'].sum() / refund_monthly['count'].sum() * 100:.2f}%")


## Revenue by product category

Not all categories are equal in terms of revenue contribution. This breakdown helps prioritise where to focus inventory investment, promotional spend, and category-level analysis. High-revenue categories with high discount rates are worth flagging — they might be propping up revenue with margin erosion.


In [ ]:

cat_rev = pp.groupby('category').agg(
    total_revenue=('total_revenue','sum'),
    units_sold=('units_sold','sum'),
    products=('product_id','count'),
    avg_discount_rate=('discount_rate','mean')
).sort_values('total_revenue', ascending=False)
print(cat_rev.round(2))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cat_rev['total_revenue'].plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Revenue by Category'); plt.sca(axes[0]); plt.xticks(rotation=30, ha='right')
cat_rev['units_sold'].plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Units Sold by Category'); plt.sca(axes[1]); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()


## Top 15 products — the revenue drivers

In most e-commerce businesses, a small number of products drive a disproportionate share of revenue. Knowing which products those are matters for inventory planning, promotional prioritisation, and risk management (what happens if one of these goes out of stock?).


In [ ]:

top15 = pp.nlargest(15, 'total_revenue')
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top15['product_id'].astype(str), top15['total_revenue'], color='steelblue')
ax.set_title('Top 15 Products by Net Revenue'); ax.set_xlabel('Net Revenue ($)')
ax.set_ylabel('Product ID'); plt.tight_layout(); plt.show()


## Premium vs non-premium

Premium products typically have higher margins and attract higher-intent customers. If premium is generating a disproportionate share of revenue relative to its share of the catalogue, that's a strong signal to double down on premium listings and potentially extend the range.


In [ ]:

prem = pp.groupby('is_premium').agg(
    products=('product_id','count'),
    avg_price=('base_price','mean'),
    total_revenue=('total_revenue','sum'),
    units_sold=('units_sold','sum'),
).round(2)
prem.index = ['Non-Premium', 'Premium']
print(prem)
print(f"\nPremium revenue share: {prem.loc['Premium','total_revenue'] / prem['total_revenue'].sum() * 100:.1f}%")
print(f"Premium catalogue share: {prem.loc['Premium','products'] / prem['products'].sum() * 100:.1f}%")
